# Тестирование модели детекции углов шахматной доски

Этот блокнот для тестирования обученной модели YOLO11 на ваших изображениях.

## Импорты и настройки

In [5]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import numpy as np

# Проверка доступности matplotlib
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    MATPLOTLIB_AVAILABLE = False
    print(f"Matplotlib недоступен ({e}), будет использован OpenCV для отображения")

## Загрузка модели

In [6]:
# Путь к обученной модели
MODEL_PATH = "best_board.pt"  # Замените на путь к вашей модели

print(f"Загрузка модели: {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print("Модель загружена")

Загрузка модели: best_board.pt
Модель загружена


## Загрузка изображения

In [13]:
# Путь к тестовому изображению
IMAGE_PATH = "image.jpg"  # Замените на путь к вашему изображению

# Загрузка изображения
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Изображение не найдено: {IMAGE_PATH}")

# Конвертация BGR -> RGB для matplotlib
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

h, w = image.shape[:2]
print(f"Изображение загружено: {w}x{h} пикселей")

# Показываем оригинальное изображение
if MATPLOTLIB_AVAILABLE:
    try:
        plt.figure(figsize=(12, 8))
        plt.imshow(image_rgb)
        plt.title("Оригинальное изображение")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Ошибка matplotlib: {e}")
        print("Используем OpenCV для отображения")
        cv2.imshow("Оригинальное изображение", image)
        print("Нажмите любую клавишу для продолжения...")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
else:
    cv2.imshow("Оригинальное изображение", image)
    print("Нажмите любую клавишу для продолжения...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Изображение загружено: 2560x1920 пикселей
Ошибка matplotlib: 'RcParams' object has no attribute '_get'
Используем OpenCV для отображения
Нажмите любую клавишу для продолжения...


## Запуск предсказания

In [14]:
# Предсказание
print("Запуск предсказания...")
results = model.predict(
    source=IMAGE_PATH,
    imgsz=1280,  # Размер как при обучении
    conf=0.25,
    save=False,  # Не сохраняем автоматически
)

print("Предсказание завершено")

Запуск предсказания...

image 1/1 d:\chesscast\chess-recognition\image.jpg: 960x1280 1 Chess-Board, 59.8ms
Speed: 9.1ms preprocess, 59.8ms inference, 2.1ms postprocess per image at shape (1, 3, 960, 1280)
Предсказание завершено


## Визуализация результатов

In [15]:
# Обработка результатов
result = results[0]

# Создаем копию изображения для визуализации
vis_image = image_rgb.copy()

# Имена точек
keypoint_names = ['corner_0', 'corner_1', 'corner_2', 'corner_3']

# Цвета для точек (RGB)
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]  # Красный, Зеленый, Синий, Желтый

# Проверяем наличие keypoints
if result.keypoints is not None and len(result.keypoints.data) > 0:
    kpts = result.keypoints.data[0].cpu().numpy()
    
    print(f"\n{'='*60}")
    print(f"НАЙДЕНО KEYPOINTS: {len(kpts)} точек")
    print(f"{'='*60}")
    
    # Рисуем точки на изображении
    for idx, kpt in enumerate(kpts):
        if len(kpt) >= 3:
            x, y, vis = float(kpt[0]), float(kpt[1]), float(kpt[2])
            name = keypoint_names[idx] if idx < len(keypoint_names) else f'point_{idx}'
            
            print(f"{name}: ({x:.1f}, {y:.1f}), visibility={vis:.2f}")
            
            # Рисуем точку только если visibility > 0.5
            if vis > 0.5:
                color = colors[idx % len(colors)]
                # Рисуем круг
                cv2.circle(vis_image, (int(x), int(y)), 15, color, -1)
                # Рисуем обводку
                cv2.circle(vis_image, (int(x), int(y)), 15, (255, 255, 255), 2)
                # Подписываем точку
                cv2.putText(vis_image, f"{idx}", (int(x)+20, int(y)), 
                          cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
                cv2.putText(vis_image, f"{idx}", (int(x)+20, int(y)), 
                          cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    
    # Рисуем линии между точками (контур доски)
    if len(kpts) >= 4:
        # Соединяем точки в порядке: 0->1->2->3->0
        points = []
        for kpt in kpts:
            if len(kpt) >= 3 and kpt[2] > 0.5:
                points.append((int(kpt[0]), int(kpt[1])))
        
        if len(points) == 4:
            # Рисуем контур доски
            pts = np.array(points, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(vis_image, [pts], True, (255, 255, 255), 2)
            cv2.polylines(vis_image, [pts], True, (0, 255, 255), 1)
    
    # Показываем результат
    if MATPLOTLIB_AVAILABLE:
        try:
            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            
            # Оригинал
            axes[0].imshow(image_rgb)
            axes[0].set_title("Оригинальное изображение", fontsize=14)
            axes[0].axis('off')
            
            # С результатами
            axes[1].imshow(vis_image)
            axes[1].set_title("Результаты детекции углов", fontsize=14)
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Ошибка matplotlib: {e}")
            print("Используем OpenCV для отображения")
            # Конвертируем RGB -> BGR для OpenCV
            vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
            
            # Показываем оригинал и результат
            cv2.imshow("Оригинальное изображение", image)
            cv2.imshow("Результаты детекции углов", vis_image_bgr)
            print("Нажмите любую клавишу для закрытия окон...")
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    else:
        # Конвертируем RGB -> BGR для OpenCV
        vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
        
        # Показываем оригинал и результат
        cv2.imshow("Оригинальное изображение", image)
        cv2.imshow("Результаты детекции углов", vis_image_bgr)
        print("Нажмите любую клавишу для закрытия окон...")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    
    # Легенда
    print(f"\n{'='*60}")
    print("ЛЕГЕНДА:")
    print(f"{'='*60}")
    for idx, name in enumerate(keypoint_names):
        if idx < len(kpts) and len(kpts[idx]) >= 3:
            color_name = ['Красный', 'Зеленый', 'Синий', 'Желтый'][idx]
            print(f"  {name} ({idx}) - {color_name} цвет")
    
else:
    print("Keypoints не найдены!")
    if MATPLOTLIB_AVAILABLE:
        try:
            plt.figure(figsize=(12, 8))
            plt.imshow(image_rgb)
            plt.title("Keypoints не найдены")
            plt.axis('off')
            plt.show()
        except:
            cv2.imshow("Keypoints не найдены", image)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    else:
        cv2.imshow("Keypoints не найдены", image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()


НАЙДЕНО KEYPOINTS: 4 точек
corner_0: (1142.6, 1456.0), visibility=0.97
corner_1: (1188.4, 801.0), visibility=0.97
corner_2: (1271.0, 787.1), visibility=0.94
corner_3: (1140.5, 1284.9), visibility=0.96
Ошибка matplotlib: 'RcParams' object has no attribute '_get'
Используем OpenCV для отображения
Нажмите любую клавишу для закрытия окон...

ЛЕГЕНДА:
  corner_0 (0) - Красный цвет
  corner_1 (1) - Зеленый цвет
  corner_2 (2) - Синий цвет
  corner_3 (3) - Желтый цвет


## Детальная информация о результатах

In [12]:
# Детальная информация
if result.boxes is not None and len(result.boxes) > 0:
    print(f"\n{'='*60}")
    print("BOUNDING BOX:")
    print(f"{'='*60}")
    for i, box in enumerate(result.boxes):
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].cpu().numpy()
        print(f"  Объект {i+1}:")
        print(f"    Класс: {cls}")
        print(f"    Confidence: {conf:.3f}")
        print(f"    Координаты: ({xyxy[0]:.1f}, {xyxy[1]:.1f}) -> ({xyxy[2]:.1f}, {xyxy[3]:.1f})")

if result.keypoints is not None and len(result.keypoints.data) > 0:
    print(f"\n{'='*60}")
    print("KEYPOINTS (детально):")
    print(f"{'='*60}")
    kpts = result.keypoints.data[0].cpu().numpy()
    
    for idx, kpt in enumerate(kpts):
        if len(kpt) >= 3:
            x, y, vis = float(kpt[0]), float(kpt[1]), float(kpt[2])
            name = keypoint_names[idx] if idx < len(keypoint_names) else f'point_{idx}'
            
            # Нормализованные координаты (относительно размера изображения)
            x_norm = x / w
            y_norm = y / h
            
            print(f"\n  {name} (индекс {idx}):")
            print(f"    Абсолютные координаты: ({x:.1f}, {y:.1f})")
            print(f"    Нормализованные координаты: ({x_norm:.4f}, {y_norm:.4f})")
            print(f"    Visibility: {vis:.3f}")
            print(f"    Статус: {'Видима' if vis > 0.5 else 'Не видима'}")


BOUNDING BOX:
  Объект 1:
    Класс: 0
    Confidence: 0.941
    Координаты: (125.3, 592.2) -> (1883.4, 2269.3)

KEYPOINTS (детально):

  corner_0 (индекс 0):
    Абсолютные координаты: (649.7, 1795.2)
    Нормализованные координаты: (0.3384, 0.7013)
    Visibility: 0.981
    Статус: Видима

  corner_1 (индекс 1):
    Абсолютные координаты: (678.9, 1028.6)
    Нормализованные координаты: (0.3536, 0.4018)
    Visibility: 0.978
    Статус: Видима

  corner_2 (индекс 2):
    Абсолютные координаты: (821.7, 955.6)
    Нормализованные координаты: (0.4280, 0.3733)
    Visibility: 0.953
    Статус: Видима

  corner_3 (индекс 3):
    Абсолютные координаты: (667.3, 1617.8)
    Нормализованные координаты: (0.3476, 0.6320)
    Visibility: 0.974
    Статус: Видима


## Сохранение результата

In [11]:
# Сохранение результата
if result.keypoints is not None and len(result.keypoints.data) > 0:
    output_path = f"result_{Path(IMAGE_PATH).stem}.jpg"
    
    # Конвертируем RGB -> BGR для сохранения через OpenCV
    vis_image_bgr = cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, vis_image_bgr)
    
    print(f"Результат сохранен: {output_path}")
else:
    print("Нечего сохранять - keypoints не найдены")

Результат сохранен: result_test.jpg
